In [1]:
import numpy as np
import pandas as pd

books = pd.read_csv('books.csv')
books_sales = pd.read_csv('books_sales.csv')

[Questions](https://pythonai211225-rgb.github.io/python_libs/08_rehersal/02-pandas.html)

# Pandas Rehearsal - Deduplicate Books & Fix Sales

Two CSV files describe a bookstore. The books.csv file lists books, but some appear twice.  
The books_sales.csv file records sales that point at books by their id.  
Your job: remove the duplicate books, then repair the sales file so every sale points at the correct remaining book

`duplicated()`
`isna() / notna()`
`merge()`
`map() / replace()`

# The Task
#### What you need to do:

1. Some books appear in duplicate.  
   You can spot a duplicate because the same catalog_number shows up twice in books.csv  
   Remove duplicate books.  
   For each duplicated catalog_number, delete the row that has more empty (NaN) fields.  
   Keep the row that is more complete
3. Fix the sales file.
   In books_sales.csv, every sale that pointed at a deleted book_id must now point at the book_id that was kept

1. df books duplicates, and group by catalog_number
2. for loop with count (counts valid numbers - higher stays), or isna()/isnull() .sum() to count null values (lower number stays)
   can also use dropna(thresh=i) with a while loop on len(df_groupby_duplicated) being > 1
3. save the book ids being deleted, and the book ids being saved, in a nested list [deleted, saved]
4. use replace, [for x[0] in list_of_replacements], [for x[1] in list_of_replacements]

In [8]:
# 1 find all duplicates and sort by catalog number
books_duplicates = books[books.duplicated(subset=['catalog_number'], keep=False)]
books_duplicates
# no triplets detected. continuing under the assumption that there will be only 2 inputs for each duplicate.
# in extremely large dataframes, with many duplicates and even triplicates, this code will not work

,book_id,author_name,title,publication_year,catalog_number
32,53783,NaN,Broken Chains,1982.0,CAT-2001
33,20274,James Mejia,Broken Chains - Special Edition,1982.0,CAT-2001
34,50753,Tiffany Davenport,Paths Unseen,2009.0,CAT-2002
35,58871,Tiffany Davenport,NaN,2009.0,CAT-2002
36,86910,Timothy Trujillo,The Hidden Truth,NaN,CAT-2003
37,77386,Timothy Trujillo,The Hidden Truth - Special Edition,1983.0,CAT-2003
38,58633,NaN,NaN,1984.0,CAT-2004
39,44065,Victor Burke,The Edge of Reality - Special Edition,1985.0,CAT-2004
40,45824,NaN,Winds of Change,1994.0,CAT-2005
41,87457,Tiffany Mcgee,Winds of Change - Special Edition,1995.0,CAT-2005


In [3]:
# 2 count null values and reorder
books_duplicates['null_count'] = books_duplicates.isna().sum(axis=1)
books_duplicates.sort_values(by=['catalog_number', 'null_count'], ascending=[True, True], inplace=True)
books_duplicates

,book_id,author_name,title,publication_year,catalog_number,null_count
33,20274,James Mejia,Broken Chains - Special Edition,1982.0,CAT-2001,0
32,53783,NaN,Broken Chains,1982.0,CAT-2001,1
34,50753,Tiffany Davenport,Paths Unseen,2009.0,CAT-2002,0
35,58871,Tiffany Davenport,NaN,2009.0,CAT-2002,1
37,77386,Timothy Trujillo,The Hidden Truth - Special Edition,1983.0,CAT-2003,0
36,86910,Timothy Trujillo,The Hidden Truth,NaN,CAT-2003,1
39,44065,Victor Burke,The Edge of Reality - Special Edition,1985.0,CAT-2004,0
38,58633,NaN,NaN,1984.0,CAT-2004,2
41,87457,Tiffany Mcgee,Winds of Change - Special Edition,1995.0,CAT-2005,0
40,45824,NaN,Winds of Change,1994.0,CAT-2005,1


In [4]:
# 3 save copies to be saved and deleted separately
original = books_duplicates.iloc[0::2]['book_id']
duplicate = books_duplicates.iloc[1::2]['book_id']
replace_map = pd.Series(data=original.values, index=duplicate.values)
# first one to show up in the newly ordered duplicates df gets put in original, and the second in duplicates

In [5]:
# 4 book sales replace and clean original dataframe
books_sales_clean=books_sales.copy()
books_sales_clean['book_id'] = books_sales_clean['book_id'].replace(replace_map)
books_clean = books[~books['book_id'].isin(duplicate)].reset_index()

## Everything together:

In [6]:
import numpy as np
import pandas as pd

books = pd.read_csv('books.csv')
books_sales = pd.read_csv('books_sales.csv')

books_duplicates = books[books.duplicated(subset=['catalog_number'], keep=False)]

books_duplicates['null_count'] = books_duplicates.isna().sum(axis=1)
books_duplicates.sort_values(by=['catalog_number', 'null_count'], ascending=[True, True], inplace=True)

original = books_duplicates.iloc[0::2]['book_id']
duplicate = books_duplicates.iloc[1::2]['book_id']
replace_map = pd.Series(data=original.values, index=duplicate.values)

books_sales_clean = books_sales.copy()
books_sales_clean['book_id'] = books_sales_clean['book_id'].replace(replace_map)
books_clean = books[~books['book_id'].isin(duplicate)].reset_index()

## For cases with multiple lines of the same book:

In [7]:
import numpy as np
import pandas as pd

books = pd.read_csv('books.csv')
books_sales = pd.read_csv('books_sales.csv')

books_duplicates = books[books.duplicated(subset=['catalog_number'], keep=False)]

books_duplicates['null_count'] = books_duplicates.isna().sum(axis=1)
books_duplicates.sort_values(by=['catalog_number', 'null_count'], ascending=[True, True], inplace=True)

keep_ids = []
duplicate_ids = []
for catalog_num, group in books_duplicates.groupby('catalog_number'):
    keep_id = group['book_id'].iloc[0]
    drop_ids = group['book_id'].iloc[1:]
    
    for drop_id in drop_ids:
        keep_ids.append(keep_id)
        duplicate_ids.append(drop_id)

books_sales_clean = books_sales.copy()
books_sales_clean['book_id'] = books_sales_clean['book_id'].replace(duplicate_ids, keep_ids)
books_clean = books[~books['book_id'].isin(duplicate_ids)].reset_index(drop=True)